## 1. Load Data

In [1]:
import pandas as pd
import torch
import numpy as np
import unicodedata
from transformers import AutoTokenizer, AutoModel
from collections import defaultdict

print("Loading datasets...")
word_level_df = pd.read_csv('../data/Amirim_Project_Submission/translated_podcast_transcript_filtered.csv')

with open("../data/sentences/podcast_sentences_en.csv", "r", encoding="utf-8") as f:
    lines = f.readlines()
sentences = [line.strip().split(',', 1)[1] for line in lines[1:] if ',' in line]

print(f"Target words : {len(word_level_df)}")
print(f"Sentences    : {len(sentences)}")

/Users/YAHLIZ/miniforge3/envs/language_project_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading datasets...
Target words : 1735
Sentences    : 402


## 2. Helpers

In [2]:
def normalize(text):
    text = unicodedata.normalize('NFC', str(text))
    # Added: strip apostrophes so "didn't" -> "didnt", "Wikipedia's" -> "wikipedias"
    text = text.replace("\u2019", "").replace("'", "").replace("`", "")
    return text.strip(' .,!?"()-:;[]{}').lower()

def sentence_contains_components(sentence_lower, components):
    """Check if all phrase components appear consecutively in the sentence."""
    # Added: replace hyphens with spaces so "self-satisfied" -> ["self", "satisfied"]
    sentence_lower = sentence_lower.replace("-", " ")
    words = sentence_lower.split()
    words_norm = [normalize(w) for w in words]
    for i in range(len(words_norm) - len(components) + 1):
        if all(words_norm[i+j] == components[j] for j in range(len(components))):
            return True
    return False

def get_sentence_tokens(sentence, tokenizer, model):
    encoded = tokenizer(sentence, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**encoded)
    token_embeddings = outputs.last_hidden_state.squeeze(0)
    word_ids = encoded.word_ids()
    input_ids = encoded['input_ids'][0]

    word_vectors = {}
    word_token_ids = {}
    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        if word_id not in word_vectors:
            word_vectors[word_id] = []
            word_token_ids[word_id] = []
        word_vectors[word_id].append(token_embeddings[idx].numpy())
        word_token_ids[word_id].append(input_ids[idx].item())

    tokens = []
    for word_id in sorted(word_vectors.keys()):
        avg_vector = np.mean(word_vectors[word_id], axis=0)
        raw_text = tokenizer.decode(word_token_ids[word_id])
        norm_text = normalize(raw_text)
        if norm_text:
            tokens.append({"text": norm_text, "vector": avg_vector})
    return tokens

## 3. Assign Words to Sentences

In [3]:
print("Assigning target words to sentences...")

word_to_sentence = {}
unassignable = []
assignment_counts = defaultdict(lambda: defaultdict(int))

sent_idx = 0
for wi in range(len(word_level_df)):
    target_raw = str(word_level_df.iloc[wi]['en']).strip().lower()
    components = [c.strip() for c in target_raw.split('_')]
    target_key = '_'.join(components)

    found = False
    for lookahead in range(min(6, len(sentences) - sent_idx)):
        candidate_idx = sent_idx + lookahead
        
        sent_words = [normalize(w) for w in sentences[candidate_idx].replace("-", " ").replace("%", " percent ").split()]        
        
        capacity = 0
        i = 0
        while i <= len(sent_words) - len(components):
            if all(sent_words[i+j] == components[j] for j in range(len(components))):
                capacity += 1
                i += len(components)
            else:
                i += 1
        
        already_assigned = assignment_counts[candidate_idx][target_key]
        
        if capacity > already_assigned:
            word_to_sentence[wi] = candidate_idx
            assignment_counts[candidate_idx][target_key] += 1
            sent_idx = candidate_idx
            found = True
            break

    if not found:
        unassignable.append(wi)
        word_to_sentence[wi] = None

assigned = sum(1 for v in word_to_sentence.values() if v is not None)
print(f"Assigned : {assigned} / {len(word_level_df)}")
print(f"Dropped  : {len(unassignable)}")

print("\nFirst 10 assignments:")
for wi in range(10):
    si = word_to_sentence[wi]
    sent_preview = sentences[si][:70] if si is not None else "UNASSIGNED"
    print(f"  [{wi:3d}] '{word_level_df.iloc[wi]['en']}'  -> sent {si}: '{sent_preview}'")

Assigning target words to sentences...
Assigned : 1481 / 1735
Dropped  : 254

First 10 assignments:
  [  0] 'act'  -> sent 0: 'Act One, Monkey in the Middle.'
  [  1] 'monkey'  -> sent 0: 'Act One, Monkey in the Middle.'
  [  2] 'middle'  -> sent 0: 'Act One, Monkey in the Middle.'
  [  3] 'places'  -> sent 1: 'So there's some places where animals almost never go, places that are '
  [  4] 'animals'  -> sent 1: 'So there's some places where animals almost never go, places that are '
  [  5] 'go'  -> sent 1: 'So there's some places where animals almost never go, places that are '
  [  6] 'places'  -> sent 1: 'So there's some places where animals almost never go, places that are '
  [  7] 'designed'  -> sent 1: 'So there's some places where animals almost never go, places that are '
  [  8] 'humans'  -> sent 1: 'So there's some places where animals almost never go, places that are '
  [  9] 'humans'  -> sent 1: 'So there's some places where animals almost never go, places that are '


In [4]:
# --- 3b. RESCUE: full scan for unassigned words, no pointer ---
print(f"Rescuing {len(unassignable)} unassigned words via full scan...\n")

# Build time map from successfully assigned words
sentence_time_map = {}
for wi, si in word_to_sentence.items():
    if si is None:
        continue
    t_start = word_level_df.iloc[wi]['start']
    t_end = word_level_df.iloc[wi]['end']
    if si not in sentence_time_map:
        sentence_time_map[si] = [t_start, t_end]
    else:
        sentence_time_map[si][0] = min(sentence_time_map[si][0], t_start)
        sentence_time_map[si][1] = max(sentence_time_map[si][1], t_end)

rescued = 0
still_unassignable = []

for wi in unassignable:
    target_raw = str(word_level_df.iloc[wi]['en']).strip().lower()
    components = [c.strip() for c in target_raw.split('_')]
    target_key = '_'.join(components)
    t = word_level_df.iloc[wi]['start']
    
    candidates = []
    for si in range(len(sentences)):
        sent_words = [normalize(w) for w in 
                      sentences[si].replace("-", " ")
                                   .replace("%", " percent ")
                                   .split()]
        capacity = 0
        i = 0
        while i <= len(sent_words) - len(components):
            if all(sent_words[i+j] == components[j] for j in range(len(components))):
                capacity += 1
                i += len(components)
            else:
                i += 1
        
        remaining = capacity - assignment_counts[si][target_key]
        if remaining > 0:
            if si in sentence_time_map:
                sent_t = sentence_time_map[si][0]
                proximity = abs(sent_t - t)
            else:
                proximity = 9999
            candidates.append((proximity, si))
    
    if candidates:
        candidates.sort()
        best_si = candidates[0][1]
        word_to_sentence[wi] = best_si
        assignment_counts[best_si][target_key] += 1
        rescued += 1
    else:
        still_unassignable.append(wi)

print(f"Rescued via full scan : {rescued}")
print(f"True unassignable     : {len(still_unassignable)}")
unassignable = still_unassignable

if still_unassignable:
    print(f"\nWords with no matching sentence anywhere:")
    for wi in still_unassignable:
        row = word_level_df.iloc[wi]
        print(f"  [{wi:4d}] t={row['start']:.1f}s  '{row['en']}'")

Rescuing 254 unassigned words via full scan...

Rescued via full scan : 225
True unassignable     : 29

Words with no matching sentence anywhere:
  [  50] t=54.0s  'mean'
  [ 165] t=151.2s  'trying'
  [ 291] t=270.0s  'last_years'
  [ 329] t=307.1s  'wanting'
  [ 418] t=397.7s  'fair_use'
  [ 427] t=409.1s  'alls'
  [ 677] t=643.6s  'cause'
  [ 807] t=761.4s  'created'
  [ 894] t=858.1s  'need'
  [ 934] t=910.0s  'cause'
  [ 953] t=931.3s  'know'
  [1039] t=1021.8s  'beaching'
  [1089] t=1073.7s  'dash'
  [1095] t=1081.6s  'afternoon'
  [1098] t=1083.2s  'federal_court'
  [1100] t=1085.6s  'unanticipated'
  [1140] t=1128.1s  'answered'
  [1177] t=1163.7s  'fourteenth'
  [1195] t=1181.2s  'endgame'
  [1298] t=1306.8s  'know'
  [1404] t=1424.8s  'cause'
  [1408] t=1430.6s  'part'
  [1447] t=1481.6s  'appeals_court'
  [1462] t=1494.5s  'honor'
  [1470] t=1504.4s  'case'
  [1480] t=1512.8s  'pleading'
  [1481] t=1513.7s  'article'
  [1576] t=1614.0s  'discussed'
  [1578] t=1618.1s  'granti

## 4. Load Model

In [5]:
print("Loading XLM-RoBERTa...")
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModel.from_pretrained('xlm-roberta-base')
model.eval()

Loading XLM-RoBERTa...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]Error processing line 1 of /Users/YAHLIZ/miniforge3/envs/language_project_env/lib/python3.10/site-packages/distutils-precedence.pth:

  Traceback (most recent call last):
    File "/Users/YAHLIZ/miniforge3/envs/language_project_env/lib/python3.10/site.py", line 195, in addpackage
      exec(line)
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named '_distutils_hack'

Remainder of file ignored
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9266.56it/s]
XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; no

XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=Tru

## 5. Extraction Loop

In [6]:
print("Extracting embeddings...\n")

final_aligned_embeddings = []
matched_word_indices = []
dropped_word_indices = []

sentence_token_cache = {}

def get_tokens_cached(si):
    if si not in sentence_token_cache:
        sentence_token_cache[si] = get_sentence_tokens(sentences[si], tokenizer, model)
    return sentence_token_cache[si]

words_per_sentence = defaultdict(list)
for wi, si in word_to_sentence.items():
    if si is not None:
        words_per_sentence[si].append(wi)

for si in sorted(words_per_sentence.keys()):
    target_word_indices = sorted(words_per_sentence[si])
    sentence_tokens = get_tokens_cached(si)
    n_tokens = len(sentence_tokens)

    consumed_positions = set()
    sentence_pointer = 0

    for word_idx in target_word_indices:
        target_raw = str(word_level_df.iloc[word_idx]['en']).strip().lower()
        target_components = [c.strip() for c in target_raw.split('_')]
        phrase_len = len(target_components)

        def try_match_at(start):
            if start + phrase_len > n_tokens:
                return False
            if set(range(start, start + phrase_len)) & consumed_positions:
                return False
            return all(
                sentence_tokens[start + i]['text'] == target_components[i]
                for i in range(phrase_len)
            )

        # Pass 1: forward from current pointer
        matched_at = None
        for start in range(sentence_pointer, n_tokens):
            if try_match_at(start):
                matched_at = start
                break

        # Pass 2: backward (handles repeated words)
        if matched_at is None:
            for start in range(0, sentence_pointer):
                if try_match_at(start):
                    matched_at = start
                    break

        if matched_at is not None:
            phrase_vectors = [sentence_tokens[matched_at + i]['vector']
                              for i in range(phrase_len)]
            final_aligned_embeddings.append(np.mean(phrase_vectors, axis=0))
            matched_word_indices.append(word_idx)
            for i in range(phrase_len):
                consumed_positions.add(matched_at + i)
            if matched_at >= sentence_pointer:
                sentence_pointer = matched_at + phrase_len
        else:
            dropped_word_indices.append(word_idx)

Extracting embeddings...



## 6. Report

In [7]:
print("=" * 55)
print("FINAL ALIGNMENT REPORT")
print("=" * 55)

unassigned_count = len(unassignable)
extracted_count = len(final_aligned_embeddings)
dropped_in_extraction = len(dropped_word_indices)

print(f"Total target words       : {len(word_level_df)}")
print(f"Assigned to a sentence   : {len(word_level_df) - unassigned_count}")
print(f"Unassigned (no sentence) : {unassigned_count}")
print(f"Successfully matched     : {extracted_count}")
print(f"Dropped in extraction    : {dropped_in_extraction}")
print(f"Total accounted for      : {extracted_count + dropped_in_extraction + unassigned_count}")
print(f"Match rate (of total)    : {extracted_count/len(word_level_df)*100:.1f}%")

if dropped_word_indices:
    print(f"\nDropped in extraction:")
    for idx in dropped_word_indices:
        row = word_level_df.iloc[idx]
        si = word_to_sentence.get(idx)
        print(f"  [{idx:4d}] '{row['en']}'  -> sent {si}: '{sentences[si][:60] if si else 'NONE'}'")

if unassignable:
    print(f"\nFirst 20 unassigned words (no sentence found):")
    for wi in unassignable[:20]:
        row = word_level_df.iloc[wi]
        print(f"  [{wi:4d}] t={row['start']:.1f}s  '{row['en']}'")

FINAL ALIGNMENT REPORT
Total target words       : 1735
Assigned to a sentence   : 1706
Unassigned (no sentence) : 29
Successfully matched     : 1692
Dropped in extraction    : 14
Total accounted for      : 1735
Match rate (of total)    : 97.5%

Dropped in extraction:
  [ 975] 'years'  -> sent 5: 'A few years ago, the photographer David Slater traveled ther'
  [1695] 'male'  -> sent 122: 'If you type Jimmy Wales monkey selfie into the Google, you'l'
  [ 556] 'dont_know'  -> sent 127: 'I don't know where that comes from.'
  [ 909] 'dont_know'  -> sent 212: 'I don't know how true that is.'
  [ 959] 'dont_know'  -> sent 233: 'I don't know if you've seen my LinkedIn profile.'
  [1048] 'federal_court'  -> sent 242: 'Monkeys, he said, do not have standing to sue in federal cou'
  [1109] 'common_law'  -> sent 260: 'In the vast history of common law prior to the Copyright Act'
  [1110] 'copyright_act'  -> sent 260: 'In the vast history of common law prior to the Copyright Act'
  [1163] 'own_pro

## 7. Save

In [8]:
if len(final_aligned_embeddings) > 0:
    embeddings_df = pd.DataFrame(final_aligned_embeddings)
    out_emb = '../data/processed/en_contextual_aligned_embeddings.csv'
    out_idx = '../data/processed/en_contextual_matched_indices.csv'
    embeddings_df.to_csv(out_emb, index=False)
    pd.DataFrame({'original_word_idx': matched_word_indices}).to_csv(out_idx, index=False)
    print(f"Saved embeddings -> {out_emb}  shape: {embeddings_df.shape}")
    print(f"Saved index map  -> {out_idx}")

    emb = pd.read_csv('../data/processed/en_contextual_aligned_embeddings.csv')
    idx = pd.read_csv('../data/processed/en_contextual_matched_indices.csv')

    print(f"Embeddings shape : {emb.shape}")       # should be (1441, 768)
    print(f"Index map shape  : {idx.shape}")        # should be (1441, 1)
    print(f"Index range      : {idx['original_word_idx'].min()} - {idx['original_word_idx'].max()}")
    print(f"Any NaN in embeddings: {emb.isnull().any().any()}")
    print(f"\nFirst 5 indices: {idx['original_word_idx'].tolist()[:5]}")
    print(f"Sample embedding row 0 (first 5 dims): {emb.iloc[0, :5].tolist()}")

Saved embeddings -> ../data/processed/en_contextual_aligned_embeddings.csv  shape: (1692, 768)
Saved index map  -> ../data/processed/en_contextual_matched_indices.csv
Embeddings shape : (1692, 768)
Index map shape  : (1692, 1)
Index range      : 0 - 1734
Any NaN in embeddings: False

First 5 indices: [0, 1, 2, 3, 4]
Sample embedding row 0 (first 5 dims): [0.042788513, 0.047795862, 0.008115685, 0.005766019, 0.022064418]
